# Step 4 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew` running `process=Process.sequential` — the same orchestration this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files. This is multi-agent in the simplest sense: two specialized roles in a fixed pipeline, not dynamic delegation.

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., Bansal, G., Zhang, J., Wu, Y., Li, B., Zhu, E., Jiang, L., Zhang, X., Zhang, S., Liu, J., Awadallah, A. H., White, R. W., Burger, D., & Wang, C. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

![AutoGen: conversable agents solving tasks in joint or hierarchical patterns](../assets/autogen-wu2023-fig1.png)
*Figure 1 from Wu et al. (2023). Reproduced for educational use in this course.*

## The exercise

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential` — the same orchestration `src/research_crew/crew.py` uses, just assembled directly in Python instead of via `@CrewBase`/YAML config. The second task's `context=[research_task]` is what chains them: CrewAI feeds the first task's output into the second automatically, the same idea as step 2c's chain prompting but wired declaratively instead of by hand.

This example reuses step 3's Researcher unchanged, and adds a new **Analyst** role — the same pairing this repo's full demo crew uses — that turns the Researcher's findings into a report aimed at one specific audience (a product team shipping an AI-powered hiring tool), a job the Researcher alone was never asked to do.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── The Crew — same sequential process as src/research_crew/crew.py ──────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

## Your task

1. Run the cell. Compare the Analyst's report to step 3's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Now swap in your own team's topic — keep the Researcher agent from step 3, and give it a second, genuinely complementary role and task suited to your topic.

4. Fill in the **Step 4** section of `EVALUATION.md`.

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. Give it its own `Task` with `context=[analysis_task]`, add the agent to `agents=[...]` and the task to `tasks=[...]`. Does the output meaningfully change? If it doesn't, ask yourself why.